# DataMate trên Colab Pro — chỉ sinh SQL

Notebook này dùng **Qwen3.5-9B chạy cục bộ** để sinh kế hoạch JSON `{type, sql, reason}`. `SQLSolver` thực thi câu SQL đọc dữ liệu trong DuckDB rồi trả đáp án. Dữ liệu cuộc thi nằm trên Google Drive cá nhân; **không đưa dữ liệu lên GitHub**.

Chạy các ô theo thứ tự. Chọn GPU ở **Runtime → Change runtime type**. A100 có thể nạp bf16; L4/T4 tự nén 4-bit khi thiếu VRAM. Đọc thời gian mỗi câu sau smoke test trước khi tăng phạm vi đo. Đường SQL với Qwen local cần được đo thực tế; mã SQL hiện được kiểm chủ yếu với Gemini.

Các bước SFT/LoRA cũ đã bỏ vì `evaluation.run_eval --export-sft` chỉ hỗ trợ pipeline Python, không xuất dữ liệu huấn luyện SQL. Notebook chỉ gọi `--query-engine sql`.


In [ ]:
from dataclasses import dataclass
from pathlib import Path
import base64
import getpass
import json
import os
import subprocess
import sys


@dataclass
class ColabSQLConfig:
    repo_url: str = "https://github.com/Oszumeiii/ZAIC_2026.git"
    branch: str = "main"
    checkout_dir: str = "/content/ZAIC_2026"
    data_dir: str = "/content/drive/MyDrive/ZAIC_2026"
    out_dir: str = "/content/drive/MyDrive/datamate_sql_out"
    model: str = "Qwen/Qwen3.5-9B"
    local_engine: str = "hf"
    holdout_n: int = 6
    holdout_per_biz: int | None = 5     # None = mọi câu thuộc mỗi business holdout
    time_budget_s: int = 1800          # giới hạn cho mỗi lần đo; câu chưa chạy sẽ abstain

    def __post_init__(self):
        if self.holdout_n < 1 or self.holdout_per_biz is not None and self.holdout_per_biz < 1:
            raise ValueError("holdout_n và holdout_per_biz phải dương")
        if self.time_budget_s < 1:
            raise ValueError("time_budget_s phải dương")

    @property
    def database_dir(self) -> str:
        return str(Path(self.out_dir) / "databases")


class ColabSQLRunner:
    def __init__(self, config: ColabSQLConfig):
        self.config = config
        self.root: Path | None = None
        self.train: Path | None = None
        self.test: Path | None = None

    @staticmethod
    def _run(args: list[str], *, cwd: Path | None = None, label: str) -> None:
        print(f"▶ {label}", flush=True)
        result = subprocess.run(args, cwd=cwd)
        if result.returncode:
            raise RuntimeError(f"{label} lỗi (exit code {result.returncode}); xem output ngay phía trên")

    def check_gpu(self) -> None:
        self._run(["nvidia-smi"], label="Kiểm tra GPU")
        import torch
        if not torch.cuda.is_available():
            raise RuntimeError("Không có CUDA GPU; chọn GPU trong Runtime rồi chạy lại")
        print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

    def mount_drive(self) -> None:
        from google.colab import drive
        drive.mount("/content/drive")

    def checkout(self) -> None:
        from google.colab import userdata
        try:
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = getpass.getpass("GitHub token (không lưu vào notebook): ")
        if not token:
            raise RuntimeError("Thiếu GITHUB_TOKEN trong Colab Secrets")
        auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        git = ["git", "-c", f"http.extraHeader=Authorization: Basic {auth}"]
        target = Path(self.config.checkout_dir)

        def git_run(args: list[str], cwd: Path | None = None) -> None:
            result = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
            output = (result.stdout + result.stderr).replace(token, "***").replace(auth, "***").strip()
            if output:
                print(output[-1200:])
            if result.returncode:
                raise RuntimeError("Git lỗi; kiểm tra token, quyền truy cập repo và nhánh")

        if (target / ".git").is_dir():
            git_run(git + ["fetch", "origin", self.config.branch], cwd=target)
            git_run(["git", "checkout", self.config.branch], cwd=target)
            git_run(git + ["pull", "--ff-only", "origin", self.config.branch], cwd=target)
        else:
            git_run(git + ["clone", "--branch", self.config.branch, self.config.repo_url, str(target)])
        roots = sorted({path.parent.parent for path in target.rglob("app/main.py")})
        if len(roots) != 1:
            raise RuntimeError(f"Cần đúng một thư mục chứa app/main.py; tìm thấy {len(roots)}")
        self.root = roots[0]
        if str(self.root) not in sys.path:
            sys.path.insert(0, str(self.root))
        os.chdir(self.root)
        print("ROOT:", self.root)
        self._run(["git", "log", "-1", "--oneline"], cwd=self.root, label="Commit hiện tại")

    def install_dependencies(self) -> None:
        packages = ["transformers>=5.2", "pytest", "openpyxl", "accelerate", "bitsandbytes", "peft",
                    "huggingface_hub", "duckdb>=1.4,<2", "jsonschema>=4", "xlrd>=2.0.1",
                    "python-dotenv", "google-genai>=1.0.0"]
        self._run([sys.executable, "-m", "pip", "install", "-q", "-U", *packages], label="Cài thư viện")
        result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flash-linear-attention"])
        if result.returncode:
            print("Không cài được flash-linear-attention; Qwen vẫn chạy nhưng có thể chậm hơn")

    def discover_data(self) -> None:
        base = Path(self.config.data_dir)
        if not base.is_dir():
            raise FileNotFoundError(f"Không thấy {base}; sửa data_dir trong ColabSQLConfig")
        found: dict[str, list[Path]] = {"train": [], "test": []}
        for questions in sorted(base.rglob("questions.jsonl")):
            directory = questions.parent
            kind = "train" if (directory / "answers.jsonl").is_file() else "test"
            found[kind].append(directory)
            count = sum(1 for _ in questions.open(encoding="utf-8"))
            print(f"{kind}: {directory} | {count} câu | rulebook: {(directory / 'rulebook.md').is_file()}")
        if len(found["train"]) != 1 or len(found["test"]) > 1:
            raise RuntimeError("Cần đúng một thư mục train và tối đa một thư mục public test dưới data_dir")
        self.train = found["train"][0]
        self.test = found["test"][0] if found["test"] else None
        if self.test is None:
            print("Chưa có public test: vẫn đo được trên train, chưa thể tạo submission")
        Path(self.config.out_dir).mkdir(parents=True, exist_ok=True)
        print("DuckDB/cache:", self.config.database_dir)

    def verify(self) -> None:
        if self.root is None or self.train is None:
            raise RuntimeError("Cần checkout() và discover_data() trước")
        import argparse
        from app.agent.sql_solver import SQLSolver
        from app.main import add_common_args, config_from_args
        args = add_common_args(argparse.ArgumentParser()).parse_args(
            ["--backend", "local", "--query-engine", "sql", "--database-dir", self.config.database_dir])
        configured = config_from_args(args)
        if configured.query_engine != "sql" or SQLSolver is None:
            raise RuntimeError("CLI chưa chọn đúng SQLSolver")
        print("SQLSolver + query_engine=sql: OK")

    def run_tests(self) -> None:
        self._ready()
        self._run([sys.executable, "-m", "pytest", "-q"], cwd=self.root, label="Chạy pytest")

    def _ready(self) -> None:
        if self.root is None or self.train is None:
            raise RuntimeError("Chạy checkout() và discover_data() trước")

    def _common_args(self) -> list[str]:
        return ["--backend", "local", "--query-engine", "sql",
                "--database-dir", self.config.database_dir, "--out-dir", self.config.out_dir,
                "--local-model", self.config.model, "--local-engine", self.config.local_engine]

    def _holdout_args(self) -> list[str]:
        args = ["--subset", "holdout", "--holdout-n", str(self.config.holdout_n)]
        if self.config.holdout_per_biz is not None:
            args += ["--per-biz", str(self.config.holdout_per_biz)]
        return args

    def evaluate(self, run_name: str, *, smoke: bool = False, extra: tuple[str, ...] = ()) -> None:
        self._ready()
        args = [sys.executable, "-m", "evaluation.run_eval", "--train-dir", str(self.train),
                *self._common_args(), *self._holdout_args()]
        if self.test is not None:
            args += ["--test-dir", str(self.test)]
        if smoke:
            args += ["--max-biz", "1", "--per-biz", "3"]
        args += ["--time-budget", str(self.config.time_budget_s), "--run-name", run_name, *extra]
        self._run(args, cwd=self.root, label=f"Đo SQL: {run_name}")

    def show_sql(self, limit: int = 3) -> None:
        path = Path(self.config.out_dir) / "logs/train_log.jsonl"
        if not path.is_file():
            raise FileNotFoundError("Chưa có train_log.jsonl; chạy evaluate() trước")
        with path.open(encoding="utf-8") as stream:
            for _, line in zip(range(limit), stream):
                row = json.loads(line)
                print("\nID:", row.get("id"), "| type:", row.get("type"), "| agree:", row.get("agree"))
                print("SQL:", (row.get("sql") or "")[:1200])
                print("Lỗi:", row.get("errors", [])[:2])

    def compare_holdout(self, run_names: list[str]) -> None:
        rows = []
        for name in run_names:
            path = Path(self.config.out_dir) / "runs" / f"{name}_holdout.json"
            data = json.loads(path.read_text(encoding="utf-8"))
            if data.get("subset") != "holdout":
                raise ValueError(f"{name} không phải holdout")
            rows.append(data)
        for row in sorted(rows, key=lambda item: item["pa"], reverse=True):
            print(f"{row['run_name']}: PA={row['pa']:+.4f} | SE={row['pa_stderr']:.4f} | "
                  f"n={row['n']} | đúng={row['correct']} sai={row['wrong']} abstain={row['abstain']} | "
                  f"{row['avg_seconds_per_question']} s/câu")
        if len({row["n"] for row in rows}) != 1:
            print("CẢNH BÁO: số câu khác nhau; kiểm tra time budget/phạm vi trước khi so điểm")
        print("Sai số được in để tham khảo; muốn kết luận cần tăng số business/câu holdout")

    def predict(self, run_name: str = "final_sql", *, extra: tuple[str, ...] = ()) -> None:
        self._ready()
        if self.test is None:
            raise RuntimeError("Chưa có public test; kiểm tra data_dir")
        args = [sys.executable, "-m", "app.main", "--test-dir", str(self.test),
                *self._common_args(), "--run-name", run_name, *extra]
        self._run(args, cwd=self.root, label="Sinh SQL và tạo submission cho public test")
        print("Bài nộp:", Path(self.config.out_dir) / "submission.jsonl")


## 1. Cấu hình và chuẩn bị môi trường

Sửa `data_dir` nếu thư mục Drive khác. Đặt `GITHUB_TOKEN` trong Colab Secrets và bật **Notebook access**. Nếu vừa nâng phiên bản `transformers` sau khi đã import thư viện, hãy restart session rồi chạy lại từ đầu.


In [ ]:
config = ColabSQLConfig()
runner = ColabSQLRunner(config)
runner.mount_drive()
runner.checkout()
runner.install_dependencies()
runner.check_gpu()
runner.discover_data()
runner.verify()


## 2. Kiểm tra mã nguồn

Bài toán SQL phụ thuộc ingestion, DuckDB và cấu hình CLI; pytest giúp phát hiện lỗi trước khi tải model.


In [ ]:
runner.run_tests()


## 3. Smoke test SQL

Chạy 3 câu của một business holdout. Theo dõi phần chuẩn bị DuckDB, SQL được sinh, lỗi sửa SQL và thời gian mỗi câu. Có thể mở ô khác chạy `!nvidia-smi` để xem VRAM.


In [ ]:
runner.evaluate("sql_smoke", smoke=True)
runner.show_sql()
runner.compare_holdout(["sql_smoke"])


## 4. Baseline trên business holdout

Mặc định 6 business × 5 câu. Tăng `config.holdout_per_biz` lên 15 hoặc `None` **sau khi** ước tính thời gian từ smoke test. Mỗi lần chạy có cache riêng theo `run_name`; chạy lại cùng tên để tiếp tục sử dụng kết quả đã cache. `time_budget_s` giới hạn mỗi lần đo.


In [ ]:
runner.evaluate("qwen35_sql")
runner.show_sql()
runner.compare_holdout(["qwen35_sql"])


## 5. Thử thay đổi trên cùng holdout

`--no-hints` tác động đến context SQL; tăng `n_samples`/`agree_min` đổi quy tắc đồng thuận. Các cờ của prompt sinh mã Python như `--no-helpers` không được dùng ở đây. So sánh các lần chạy cùng phạm vi và cùng giới hạn thời gian.


In [ ]:
runner.evaluate("sql_no_hints", extra=("--no-hints",))
runner.evaluate("sql_n5", extra=("--n-samples", "5", "--agree-min", "3"))
runner.compare_holdout(["qwen35_sql", "sql_no_hints", "sql_n5"])


## 6. Dự đoán public test

Chỉ chạy khi đã chọn cấu hình từ holdout. Đặt `FINAL_FLAGS` theo biến thể thắng ở bước 5: `()` cho baseline, `("--no-hints",)` hoặc `("--n-samples", "5", "--agree-min", "3")`. Lần chạy public test không có time budget để xử lý hết câu; lệnh sẽ ghi và kiểm định `submission.jsonl` trong Drive. Không có đáp án public test để tính PA tại đây.


In [ ]:
FINAL_FLAGS = ()  # thay theo kết quả holdout, nếu cần
runner.predict(extra=FINAL_FLAGS)
